### Integer Division and Modulo Through Repeated Subtraction with SuperNeuroMAT

**Tutorial Author and Code Implementation:** Simon Weston <br> 
**Algorithmic and Architectural Design:** Simon Weston (feedback, quotient, and remainder logic), conceptual contributions from Duncan Rowden (iterative division architecture) <br>
**Base Structural Influence:** *Virtual Neuron: A Neuromorphic Approach for Encoding Numbers* (Date et al., 2023) <br>

In this tutorial we'll build on our existing two's complement subtraction system to create a generalized spiking neural network that divides two integers, producing a quotient and a remainder. The system takes in two positive n-digit binary inputs and produces a binary remainder and a temporally encoded quotient.

#### System Structure

![network diagram](img/integer_division_and_modulo_images/Division_Diagram.jpg)

**Diagram 1:** Modified by Simon Weston, base structure from *Virtual Neuron: A Neuromorphic Approach for Encoding Numbers* (Date et al., 2023)

When we divide two numbers, we're seeing how many times a divisor can fit into a dividend. In other words, we're repeatedly subtracting a divisor from a dividend, stopping when we get a result less than 0, then counting how many times we've subtracted to find the quotient. We can also check the result of the last successful subtraction operation to find out how much remainder we have left. Our network models this through the following modifications to the existing two's compliment subtraction network (see tutorial #13): <br>
<br>
1. **Adding a set of neurons that will repeatedly send the divisor's two's complement to the hidden layer (~x1_repeat and ~x2_repeat):** <br>
Since our original inverter neurons (~x0 and ~x1 in the diagram) have infinite refractory period, we need to store the divisor's binary value in another set, hence the creation of ~x1_repeat and ~x0_repeat. These neurons, along with the add one neuron, have autaptic (self-connecting) synapses. Together, this structure will send a constant value to be taken away from the changing dividend on each iteration. <br>
<br>
2. **Adding output feedback synapses that recycle the new dividend after each subtraction iteration:** <br>
In the diagram, we have synapses connecting z_0 and z_1 directly back into the hidden layer to maximize time efficiency. After our first iteration of subtraction, these become our new input, replacing y_0 and y_1 and representing our changing dividend after each round of subtracton. We'll plan synapse delays such that all spikes (both dividend and divisor) arrive to the hidden layer at the same intervals.<br>
<br>
3. **Adding a temporally spiking quotient neuron:** <br>
The purpose of this neuron (top right of diagram) is to constantly spike coinciding with each successful subtraction iteration so that its number of spikes equals our quotient. We accomplish this by creating an autaptic synapse and setting its delay to the time each iteration takes. We need the neuron to stop counting when the divisor is greater than the dividend. This is done by creating an inhibitory synapse with an extremely negative weight (red arrow) between the sign bit, which spikes when we get a negative result, and the quotient neuron. <br>
<br>
4. **Adding a binary remainder layer:**<br>
The purpose of the remainder layer is to mirror the result of each successful subtraction iteration, stopping when we get a negative value (divisor > dividend). We accomplish this by connecting a layer of neurons to each z output and creating an inhibitory synapse with extremely negative weight (red arrows) between the sign bit and each remainder neuron.<br>

#### Step 1: Set Up and Creating the Divisor's Repeating Layer 

In [85]:
#If you haven't yet installed SuperNeuroMat, copy the following and run it in your terminal: 
#pip install superneuromat

Since we have an existing subtraction network utilizing two's complement representation (see tutorial #13), we'll start by taking a copy of it for reference and renaming it:

In [86]:
import superneuromat as snm

def division(bit_count):

    DIV = snm.SNN() # short for division


    x_inverted = []
    x_input = []
    y_input = []
    bit_groups = []
    z_outputs = []

    # create inputs and twos complement representation of x
    for i in range(bit_count):
        x_input.append(DIV.create_neuron(threshold=0))

    for i in range(bit_count):
        y_input.append(DIV.create_neuron(threshold=0))

    for i in range(bit_count):
        neuron = DIV.create_neuron(
            threshold=0,
            leak=1,
            reset_state=1/2,
            initial_state=-3/2,
            refractory_period=1000
        )
        x_inverted.append(neuron)

    add_one = DIV.create_neuron(threshold=0)


    # create hidden layer
    for bit in range(bit_count):

        cluster = []

        thresholds = [0, 1, 2]

        for threshold in thresholds:
            neuron = DIV.create_neuron(threshold=threshold)
            cluster.append(neuron)

        bit_groups.append(cluster)


    # create output neurons
    for i in range(bit_count):
        z_outputs.append(DIV.create_neuron(threshold=0))

    # input to inverted synapses
    for i in range(bit_count):
        DIV.create_synapse(
            x_input[i],
            x_inverted[i],
            weight=-1000,
        )
        
    # input to hidden layer synapses
    for bit in range(bit_count):
        
        for neuron in bit_groups[bit]:
            DIV.create_synapse(
                pre_id = x_inverted[bit],
                post_id = neuron,
                weight = 1.0,
                delay = bit + 1          
            )

            DIV.create_synapse(      
                pre_id = y_input[bit],    
                post_id = neuron,
                weight=1.0,
                delay = bit + 2          
            )

    for neuron in bit_groups[0]:
            DIV.create_synapse(add_one, neuron, delay = 2)

            
    # carry over synapses
    for bit in range(bit_count - 1):

        carry_neuron = bit_groups[bit][1]

        for neuron in bit_groups[bit + 1]:
            DIV.create_synapse(
                carry_neuron,
                neuron,
                weight = 1.0,
                delay = 1
            )

    # hidden to output layer synapses
    for bit in range(bit_count):
        
        delay = bit_count - bit
                
        threshold_0 = bit_groups[bit][0]
        threshold_1 = bit_groups[bit][1]
        threshold_2 = bit_groups[bit][2]

        DIV.create_synapse(
            pre_id = threshold_0,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )

        DIV.create_synapse(
            pre_id = threshold_1,
            post_id = z_outputs[bit],
            weight = -1.0,
            delay = delay
        )

        DIV.create_synapse(
            pre_id = threshold_2,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )

    return DIV, x_input, y_input, x_inverted, z_outputs, bit_groups, add_one

BIT_COUNT = 3

DIV, x_input, y_input, x_inverted, z_outputs, bit_groups, add_one = (
    division(BIT_COUNT)
)


In [87]:
# test section
y_value = 3
x_value = 1

DIV.add_spike(0, add_one)

for bit in range(BIT_COUNT):
    if (x_value >> bit) & 1:
        DIV.add_spike(0, x_input[bit])

    if (y_value >> bit) & 1:
        DIV.add_spike(0, y_input[bit])

DIV.simulate(BIT_COUNT + 3)

### Decoding Section: 

# fnding output in binary
output_bits = []

for neuron in z_outputs:
    neuron_id = neuron.idx

    spiked = any(
        DIV.spike_train[t][neuron_id]
        for t in range(len(DIV.spike_train))
    )

    if spiked:
        output_bits.append(1)
    else:
        output_bits.append(0)

binary_output = output_bits[::-1]

print("\nOutput bits:")
print(binary_output)


# converting binary output to base-ten:
result = 0

for bit in binary_output:
    result = result * 2 + bit

# interpreting two's complement output:
if binary_output[0] == 1:
    result -= 2 ** len(binary_output)

print("Expected:", y_value - x_value)
print("\nResult:", result)

print("\nSpike train:")
DIV.print_spike_train()



if result == y_value - x_value:
        print("Correct")
else:
        print("Incorrect")



Output bits:
[0, 1, 0]
Expected: 2

Result: 2

Spike train:
t|id0 1 2 3 4 5 6 7 8 9 101112131415161718192021222324252627282930313233343536373839404142434445464748495051525354555657585960 
0: [├─│ │ ├─├─│ │ │ │ ├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ]
1: [│ │ │ │ │ │ │ ├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ ├─├─├─│ ├─│ │ ├─│ │ ├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─├─├─│ │ │ │ │ │ │ │ │ ]
2: [│ │ │ │ │ │ │ │ │ │ ├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ ├─│ ├─├─│ ├─├─│ ├─├─│ │ │ │ ├─│ │ │ │ ├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ]
3: [│ │ │ │ │ │ │ │ │ │ │ │ │ ├─├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─│ │ │ │ ├─│ │ │ │ ├─│ │ │ │ │ │ ├─│ ├─│ │ │ │ │ │ ]
4: [│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─│ ├─│ │ ├─├─├─]
5: [│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ]
Correct


##### Repeating the Inverted Divisor

Let's create x_inverted's repeating layer. This layer receives the first inverted copy from x_inverted then continues spiking on interval with autaptic synapses. 

In [88]:
x_inverted_repeater = []

# creating x_inverted_repeater layer
for i in range(bit_count):
    neuron = DIV.create_neuron(threshold=0)
    x_inverted_repeater.append(neuron)

# first copy of x_inverted arrives at the repeater
    DIV.create_synapse(
        x_inverted[i],
        neuron,
        delay=bit_count+1
    )
   # then keeps repeating
    DIV.create_synapse(
        neuron,
        neuron,
        delay=bit_count+1
    )

NameError: name 'bit_count' is not defined

Now let's send its spikes to the hidden layer (subtraction apparatus). The parameters are the same as our existing input to hidden layer synapses:

In [ ]:
for bit in range(bit_count):
    for neuron in bit_groups[bit]:
        DIV.create_synapse(
            x_inverted_repeater[bit],
            neuron,
            weight=1.0,
            delay=bit + 1
        )

And we do the same with the add_one neuron, spiking itself periodically and sending signal to subtraction network in sync with inverted x:

In [ ]:
# add_one autapses:
DIV.create_synapse(
    add_one, 
    add_one,
    delay = bit_count + 1
)

# send to hidden layer:
for neuron in bit_groups[0]:
    DIV.create_synapse(
        add_one,
        neuron,
        delay = 1
    )
    
# spike one timestep later to sync with inverted x:
DIV.add_spike(1, add_one)

#### Step 2: Feed the previous result back 

Now we'll connect z_outputs back into the hidden layer so that x_inverted_repeater's (divisor) spikes will subtract from it, using the same parameters:

In [ ]:
for bit in range(bit_count):
    for neuron in bit_groups[bit]:
        DIV.create_synapse(
            z_outputs[bit],
            neuron,
            weight=1.0,
            delay=bit+1
        )

#### Step 3: Create Temporally Spiking Quotient Neuron

The quotient neuron is a simple self-spiking neuron that will count each iteration until it's inhibited by the sign bit.

In [ ]:
# creating quotient
quotient = DIV.create_neuron()

# autapses (in sync with z_outputs):
DIV.create_synapse(
        quotient,
        quotient,
        delay=bit_count + 1
    )

# this is when our first iteration is complete:
DIV.add_spike(
        time = bit_count + 3,
        neuron_id = quotient
    )

Creating the inhibitory synapse from the sign bit so quotient stops counting when division is unsucessful:

In [ ]:
# sign bit is the last z_output neuron

DIV.create_synapse(
    z_outputs[-1],
    quotient,
        weight = -1000,
        delay = 1
    )


#### Step 4: Creating Binary Remainder Layer

The remainder layer mirrors z_outputs but is inhibited by the sign bit, so its latest spikes will denote the binary remainder.

In [ ]:
remainder=[]

# range is one less than z_outputs because there's no sign neuron

for i in range(len(z_outputs)-1):
    id = DIV.create_neuron(
        threshold = 0,
        leak = 1,
    )
    remainder.append(id)

Creating synapses to get spikes from z_outputs:

In [ ]:
for i, neuron in enumerate(z_outputs[:-1]):
    DIV.create_synapse(
        neuron,
        remainder[i],
        weight=1,
        delay=1,
    )

Lastly, we'll create the inhibitory synapses from the sign bit so the final remainder follows the last successful subtraction: 

In [ ]:
for i, neuron in enumerate(remainder):
    DIV.create_synapse(
        z_outputs[-1],
        neuron,
        weight = -1000,
        delay = 1
    )

#### Step 5: Using a function and interpreting results:

Now we'll enclose everything in a function, organize our code, and add our test, simulate, and decoding sections:

In [ ]:
import superneuromat as snm

def division(bit_count):

    DIV = snm.SNN()

    x_inverted = []
    x_input = []
    y_input = []
    bit_groups = []
    z_outputs = []
    x_inverted_repeater = []
    add_one = DIV.create_neuron()
    quotient = DIV.create_neuron()
    remainder=[]

    # ============================================================
    # neuron creation
    # ============================================================

    # create inputs and twos complement representation of x
    for i in range(bit_count):
        x_input.append(DIV.create_neuron(threshold=0))

    for i in range(bit_count):
        y_input.append(DIV.create_neuron(threshold=0))

    for i in range(bit_count):
        neuron = DIV.create_neuron(
            threshold=0,
            leak=1,
            reset_state=1/2,
            initial_state=-3/2,
            refractory_period=1000
        )
        x_inverted.append(neuron)

    # creating x_inverted_repeater layer
    for i in range(bit_count):
        neuron = DIV.create_neuron(threshold=0)
        x_inverted_repeater.append(neuron)

    # create hidden layer
    for bit in range(bit_count):

        cluster = []

        thresholds = [0, 1, 2]

        for threshold in thresholds:
            neuron = DIV.create_neuron(threshold=threshold)
            cluster.append(neuron)

        bit_groups.append(cluster)


    # create output neurons
    for i in range(bit_count):
        z_outputs.append(DIV.create_neuron(threshold=0))

    # creating remainder
    for i in range(len(z_outputs)-1):
        id = DIV.create_neuron(
            threshold = 0,
            leak = 1,
        )
        remainder.append(id)

    # ============================================================
    # synapse creation
    # ============================================================

    # getting a repeating two's complement for divisor:
    # ============================================================
    # input to inverted synapses
    for i in range(bit_count):
        DIV.create_synapse(
            x_input[i],
            x_inverted[i],
            weight=-1000,
        )

    # first copy of x_inverted arrives at the repeater
        DIV.create_synapse(
            x_inverted[i],
            x_inverted_repeater[i],
            delay=bit_count+1
        )
    # then keeps repeating
        DIV.create_synapse(
            x_inverted_repeater[i],
            x_inverted_repeater[i],
            delay=bit_count+1
        )

    # add_one autapses:
    DIV.create_synapse(
        add_one, 
        add_one,
        delay = bit_count + 1
    )

    # sending data to hidden layer:
    # ============================================================
        
    # input to hidden layer synapses
    for bit in range(bit_count):
        
        for neuron in bit_groups[bit]:
            DIV.create_synapse(
                pre_id = x_inverted[bit],
                post_id = neuron,
                weight = 1.0,
                delay = bit + 1          
            )

            DIV.create_synapse(      
                pre_id = y_input[bit],    
                post_id = neuron,
                weight=1.0,
                delay = bit + 2          
            )

            DIV.create_synapse(
                x_inverted_repeater[bit],
                neuron,
                weight=1.0,
                delay=bit + 1
        )

    for neuron in bit_groups[0]:
        DIV.create_synapse(
            add_one,
            neuron,
            delay = 1
        )

    # output to hidden layer synapses (feeding dividend back into hidden layer)
    for bit in range(bit_count):
        for neuron in bit_groups[bit]:
            DIV.create_synapse(
                z_outputs[bit],
                neuron,
                weight=1.0,
                delay=bit+1
            )
            
    # hidden layer synapses
    # ============================================================

    # carry over synapses
    for bit in range(bit_count - 1):

        carry_neuron = bit_groups[bit][1]

        for neuron in bit_groups[bit + 1]:
            DIV.create_synapse(
                carry_neuron,
                neuron,
                weight = 1.0,
                delay = 1
            )

    # hidden to z_outputs synapses
    # ============================================================
    for bit in range(bit_count):
        
        delay = bit_count - bit
                
        threshold_0 = bit_groups[bit][0]
        threshold_1 = bit_groups[bit][1]
        threshold_2 = bit_groups[bit][2]

        DIV.create_synapse(
            pre_id = threshold_0,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )

        DIV.create_synapse(
            pre_id = threshold_1,
            post_id = z_outputs[bit],
            weight = -1.0,
            delay = delay
        )

        DIV.create_synapse(
            pre_id = threshold_2,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )

    # quotient and remainder synapses (final outputs layer):
    # ============================================================
    # quotient autapse:
    DIV.create_synapse(
            quotient,
            quotient,
            delay=bit_count + 1
        )
    
    # quotient inhibitor:
    DIV.create_synapse(
        z_outputs[-1],
        quotient,
        weight = -1000,
        delay = 1
    )

    # z_outputs to remainder:
    for i, neuron in enumerate(z_outputs[:-1]):
        DIV.create_synapse(
            neuron,
            remainder[i],
            weight=1,
            delay=1,
        )

    # remainder inhibitor:
    for i, neuron in enumerate(remainder):
        DIV.create_synapse(
            z_outputs[-1],
            neuron,
            weight = -1000,
            delay = 1
        )
    
    return (DIV, x_input, y_input, x_inverted, x_inverted_repeater, add_one, bit_groups, z_outputs, quotient, remainder)



In [ ]:
# ============================================================
# test input
# ============================================================
BIT_COUNT = 8

DIV, x_input, y_input, x_inverted, x_inverted_repeater, add_one, bit_groups, z_outputs, quotient, remainder = division(BIT_COUNT)

# y / x

y_value = 127
x_value = 40


# adding spikes:

for bit in range(BIT_COUNT):

    if (x_value >> bit) & 1:

        DIV.add_spike(0, x_input[bit])

    if (y_value >> bit) & 1:
        DIV.add_spike(0, y_input[bit])

DIV.add_spike(1, add_one)

DIV.add_spike(
        time = BIT_COUNT + 3,
        neuron_id = quotient
    )

In [ ]:
# ============================================================
# simulate
# ============================================================

# Our simulation time scales with the quotient since it depends on how many successful subtraction iterations we'll have. 
max_iterations = y_value // x_value
simulation_time = (max_iterations + 2) * (BIT_COUNT + 1)
DIV.simulate(simulation_time)

In [114]:
# ============================================================
# decoding results
# ============================================================

print(f"{y_value} / {x_value}")

quotient_times = []

# go through every timestep of quotient and append time it spikes (== True)
for t in range(len(DIV.spike_train)):
    if DIV.spike_train[t][quotient.idx]:
        quotient_times.append(t)

quotient_value = len(quotient_times)

# check remainder last quotient time step
final_time = quotient_times[-1]

# binary decoding remainder, raise 2 to the power of index and sum 
remainder_value = sum(
    2**i
    for i, neuron in enumerate(remainder)
    if DIV.spike_train[final_time][neuron.idx]
)

print(f"Quotient: {quotient_value}")
print(f"Remainder: {remainder_value}")


if quotient_value == (y_value // x_value) and remainder_value == (y_value % x_value):
    print("Correct")

print(f"Simulation Time: {simulation_time}")

127 / 40
Quotient: 3
Remainder: 7
Correct
Simulation Time: 45
